# Phase 2: Data Design — Chiller Failure Prediction

**AMD Project Methodology — Phase 2**  
This notebook covers the three sub-phases of Data Design:
1. **Data Collection** — acquiring, documenting, and storing the dataset
2. **Data Understanding** — Exploratory Data Analysis with written interpretation
3. **Data Preparation** — cleaning, handling missing data, preprocessing

**Dataset:** ASHRAE RP-1043 Chiller Fault Detection Dataset  
**Source:** [Figshare DOI: 10.6084/m9.figshare.28435232.v1](https://doi.org/10.6084/m9.figshare.28435232.v1)  
**License:** CC-BY 4.0 (free for research)  
**File used:** `11000.xlsx` — 11,000 samples, 16 sensor features, 8 classes

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("All imports successful!")

---
# Part 1: Data Collection

The AMD methodology specifies seven steps for data collection. This section follows each of them.

### Step 1: Start with Data Requirements

In Phase 1 (Domain Understanding), the data requirements were defined as follows: we need continuous sensor measurements from a chiller system operating under both normal and faulty conditions, with labels identifying the fault state. The dataset must contain temperature, pressure, flow, and electrical measurements — the key variables that change when a chiller degrades or fails. At minimum, 16 fault-sensitive features are needed (as established by prior research on the RP-1043 dataset). The data must be labeled with fault type to support supervised classification.

### Step 2: Collect the Data

The dataset was downloaded from the Figshare academic data repository on 24 March 2026. The download procedure was:

1. Navigate to https://figshare.com/articles/dataset/Chiller_operation_data_set/28435232
2. Click the download button for the full dataset archive
3. Extract the archive, which contains folders per fault type, the compiled `11000.xlsx` file, and a `read me first.xls` documentation file

The `11000.xlsx` file is the pre-compiled dataset used by Guo et al. (2025) in their PLOS ONE publication. It contains 11,000 samples selected from the original ASHRAE RP-1043 experimental data, specifically the severity level 1 (SL1) faults — the mildest and hardest to detect. This is the most challenging subset, which is ideal for testing whether our model can catch incipient faults before they progress.

**Original source:** The data was originally collected by Comstock & Braun (1999) at Purdue University's Ray W. Herrick Laboratory from a 90-ton York YT centrifugal water-cooled chiller. Data was collected at 10-second intervals under controlled laboratory conditions with specific faults intentionally introduced at known severity levels.

### Step 3: Where to Store

The dataset is stored locally in the project directory under `data/`. The compiled file `11000.xlsx` is used as the primary data source. The original raw fault-specific folders from the Figshare download are kept in `data/raw/` as a backup for traceability.

### Step 4: Versioning and Naming System

| File | Description |
|------|-------------|
| `data/11000.xlsx` | Original dataset as downloaded from Figshare (untouched) |
| `data/prepared/df_clean.csv` | Cleaned and labeled version after Phase 2 processing |
| `data/prepared/X_train_scaled.npy` | Scaled training features (ready for modelling) |
| `data/prepared/data_dictionary.csv` | Feature documentation |

Version control is handled via Git. The raw `11000.xlsx` is committed as-is so the exact input data is always reproducible.

### Step 5: Retrieval Frequency

This is a static benchmark dataset — it does not update. The data was collected once in 1999 and has been used as a fixed benchmark since. No reload or update strategy is needed. In a real-world production scenario, sensor data would stream continuously from the BMS, requiring an ingestion pipeline.

### Step 6: Dataset Scope

The `11000.xlsx` file is a subset of the full RP-1043 archive. Guo et al. (2025) selected 11,000 samples: 4,000 normal operation samples and 1,000 samples per fault type (7 fault types), all at severity level 1. The original archive contains data across all four severity levels. For this project, SL1 data is used because it represents the earliest detectable stage of faults — exactly what predictive maintenance aims to catch.

### Step 7: Documentation

The full data dictionary is provided below.

### Data Dictionary

| # | Feature | Full Name | Type | Unit | Valid Range | Description | Source |
|---|---------|-----------|------|------|-------------|-------------|--------|
| 1 | TEI | Evaporator Water Inlet Temp | float64 | deg F | 43-62 | Temperature of water entering the evaporator from the building loop | Sensor (thermocouple) |
| 2 | TEO | Evaporator Water Outlet Temp | float64 | deg F | 39-52 | Temperature of chilled water leaving the evaporator (the product of the chiller) | Sensor (thermocouple) |
| 3 | TCI | Condenser Water Inlet Temp | float64 | deg F | 62-89 | Temperature of cooling water entering the condenser from the cooling tower | Sensor (thermocouple) |
| 4 | TCO | Condenser Water Outlet Temp | float64 | deg F | 65-97 | Temperature of water leaving the condenser after absorbing heat from the refrigerant | Sensor (thermocouple) |
| 5 | kW | Compressor Power | float64 | kW | 28-85 | Electrical power consumed by the compressor motor | Power meter |
| 6 | TEA | Evaporator Approach Temp | float64 | deg F | 0-12 | TEO minus TRE. Measures how efficiently heat transfers in the evaporator; rises with fouling | Derived |
| 7 | TCA | Condenser Approach Temp | float64 | deg F | 0-11 | TCO minus TRC. Measures condenser heat transfer efficiency; rises with fouling | Derived |
| 8 | TRE | Refrigerant Evaporating Temp | float64 | deg F | 35-49 | Temperature at which refrigerant evaporates inside the evaporator | Sensor (pressure-derived) |
| 9 | TRC | Refrigerant Condensing Temp | float64 | deg F | 54-104 | Temperature at which refrigerant condenses inside the condenser | Sensor (pressure-derived) |
| 10 | TRC_sub | Condenser Subcooling | float64 | deg F | 0-15 | How far below condensing temp the liquid refrigerant is cooled; drops with refrigerant leak | Derived |
| 11 | T_suc | Suction Temperature | float64 | deg F | 37-51 | Refrigerant temperature at compressor inlet | Sensor (thermocouple) |
| 12 | Tsh_suc | Suction Superheat | float64 | deg F | 0-9 | T_suc minus TRE. Rises significantly with refrigerant leak | Derived |
| 13 | TR_dis | Discharge Temperature | float64 | deg F | 79-156 | Refrigerant temperature at compressor outlet; abnormal values indicate compressor issues | Sensor (thermocouple) |
| 14 | Tsh_dis | Discharge Superheat | float64 | deg F | 0-77 | TR_dis minus TRC. Sensitive to compressor efficiency and refrigerant charge | Derived |
| 15 | TO_sump | Oil Sump Temperature | float64 | deg F | 109-123 | Temperature of lubricating oil in the compressor sump; rises with excess oil or bearing issues | Sensor (thermocouple) |
| 16 | PO_net | Net Oil Pressure | float64 | psi | 77-152 | Oil pump pressure minus suction pressure; drops indicate oil system degradation | Sensor (pressure transducer) |

**Target variable:**

| Label | Fault Type | Samples | Physical Cause |
|-------|-----------|--------|----------------|
| 1 | Normal operation | 4,000 | No fault — baseline operating conditions |
| 2 | Condenser fouling (CF) | 1,000 | Tubes plugged in the condenser, reducing heat transfer |
| 3 | Reduced condenser water flow (FWC) | 1,000 | Condenser water flow rate reduced by 10-40% |
| 4 | Reduced evaporator water flow (FWE) | 1,000 | Evaporator water flow rate reduced by 10-40% |
| 5 | Refrigerant leak (RL) | 1,000 | Refrigerant charge reduced by 10-40% below nominal |
| 6 | Refrigerant overcharge (RO) | 1,000 | Refrigerant charge increased by 10-40% above nominal |
| 7 | Excess oil (EO) | 1,000 | Oil charge increased by 14-68% above nominal |
| 8 | Non-condensables (NC) | 1,000 | Nitrogen gas introduced into the refrigerant circuit |

In [ ]:
# Load the dataset
DATA_PATH = Path("data/11000.xlsx")
df = pd.read_excel(DATA_PATH, engine="openpyxl")

LABEL_MAP = {1:'Normal', 2:'Condenser fouling', 3:'Reduced cond. water flow',
             4:'Reduced evap. water flow', 5:'Refrigerant leak',
             6:'Refrigerant overcharge', 7:'Excess oil', 8:'Non-condensables'}
LABEL_SHORT = {1:'Normal', 2:'CF', 3:'FWC', 4:'FWE', 5:'RL', 6:'RO', 7:'EO', 8:'NC'}

df['fault_name'] = df['label'].map(LABEL_MAP)
df['fault_short'] = df['label'].map(LABEL_SHORT)
FEATURE_COLS = [c for c in df.columns if c not in ['label', 'fault_name', 'fault_short']]

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
df.head()

---
# Part 2: Data Understanding (EDA)

The goal of EDA is not just to generate plots, but to **interpret what the data tells us**. Each analysis below includes a written interpretation explaining the findings and their implications for modelling.

## 2.1 Data Quality Assessment

In [ ]:
df[FEATURE_COLS].describe().round(2)

In [ ]:
total_missing = df[FEATURE_COLS].isnull().sum().sum()
n_dupes = df[FEATURE_COLS + ['label']].duplicated().sum()
print(f"Missing values: {total_missing}")
print(f"Duplicate rows: {n_dupes}")

### Interpretation — Data Quality

The dataset contains **zero missing values** across all 16 features and 11,000 samples. This is expected for a laboratory benchmark dataset where data collection was carefully controlled. In a real-world BMS deployment, missing values would be common due to sensor communication drops, power outages, or sensor malfunctions, and would require imputation strategies (forward-fill for slow-changing temperatures, interpolation for gaps under 5 minutes).

There are **3 duplicate rows**. Since the original data was collected at 10-second intervals, it is physically possible for two consecutive readings to be identical (sensors have limited resolution). These duplicates are harmless and will be removed in the Data Preparation step.

The descriptive statistics show that all features have reasonable value ranges consistent with what a centrifugal chiller would produce. For example, evaporator water outlet temperature (TEO) ranges from 39.5 to 52.1 deg F, which is typical for chilled water systems. Compressor power (kW) ranges from 27.7 to 85.0 kW, consistent with a 90-ton chiller operating across various load conditions.

## 2.2 Class Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

order = [LABEL_MAP[i] for i in range(1, 9)]
colors = ['#2ecc71'] + list(sns.color_palette('Set2', 7))
counts = df['fault_name'].value_counts().reindex(order)
counts.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Class Distribution (8 classes)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Samples')
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(counts.values):
    ax1.text(i, v + 30, f'{v:,}', ha='center', fontsize=9)

binary = pd.Series({'Normal': (df['label']==1).sum(), 'Fault (any)': (df['label']!=1).sum()})
binary.plot(kind='bar', ax=ax2, color=['#2ecc71','#e74c3c'], edgecolor='black', linewidth=0.5)
ax2.set_title('Binary Distribution', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=0)
for i, v in enumerate(binary.values):
    ax2.text(i, v + 30, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.show()
print(f"Normal:Fault ratio = {binary['Normal']/binary['Fault (any)']:.2f}:1")

### Interpretation — Class Distribution

The left chart shows all 8 classes. Normal has 4,000 samples while each of the 7 fault types has exactly 1,000 samples, totalling 7,000 fault samples.

The right chart shows the binary view (Normal vs any fault). The ratio is approximately **0.57:1** — there are actually more fault samples than normal samples.

**This is the opposite of what would occur in real-world data.** In a production data centre, a chiller might operate normally for 99%+ of the time, with failures being extremely rare events (ratio of 50:1 to 1000:1). The benchmark dataset was intentionally compiled with balanced fault representation to enable fair algorithm comparison. This is an important limitation to acknowledge: the model trained on this data may need recalibration (via `scale_pos_weight` in XGBoost or threshold adjustment) when deployed on real-world data with genuine class imbalance.

For modelling on this dataset, the moderate imbalance does not require aggressive resampling techniques like SMOTE. A `scale_pos_weight` adjustment of approximately 0.6 in XGBoost will be sufficient.

## 2.3 Feature Distributions

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(FEATURE_COLS):
    df[col].hist(bins=50, ax=axes[i], color='steelblue', edgecolor='white', linewidth=0.3)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1, label='mean')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', linewidth=1, label='median')
    axes[i].tick_params(labelsize=8)
axes[0].legend(fontsize=8)
fig.suptitle('Feature Distributions (all 16 sensor features)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Interpretation — Feature Distributions

**Approximately normal distributions:** Features like TEI, TCI, TCO, TRC, and TR_dis show roughly bell-shaped distributions. This is expected because the chiller was tested across a range of operating conditions (varying cooling load, condenser water temperature), and the operating points are spread across a realistic range.

**Skewed distributions:** Tsh_dis (discharge superheat) has a strong right skew with a long tail extending to approximately 77 deg F, which likely corresponds to specific fault conditions (particularly refrigerant leak, which dramatically increases superheat). TEA and TCA (approach temperatures) are right-skewed because approach temperatures are always positive and tend to be small under normal conditions but can increase during faults.

**Multimodal patterns:** kW (compressor power) shows a somewhat bimodal distribution, suggesting the data contains distinct operating regimes — likely low-load and high-load conditions that were tested systematically.

**Mean vs median:** For most features the mean and median are close together (red and orange dashed lines overlap), indicating relatively symmetric distributions. Where they diverge (e.g., Tsh_dis, TCA), this confirms the skewness. This is one reason we will use **RobustScaler** (based on median and IQR) rather than StandardScaler (based on mean and std) for preprocessing — it is less affected by these skewed tails.

## 2.4 Correlation Analysis

In [ ]:
corr = df[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 8},
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Highly correlated pairs (|r| > 0.85):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.85:
            print(f"  {corr.columns[i]:12s} <-> {corr.columns[j]:12s}  r = {r:+.3f}")

### Interpretation — Correlations

The correlation heatmap reveals several strong relationships, all explainable by the physics of the chiller system:

**TCO and TRC (r = +0.97):** The condenser water outlet temperature and the refrigerant condensing temperature are almost perfectly correlated. This makes physical sense — the condenser is a heat exchanger where the refrigerant and water are in direct thermal contact, so their temperatures track each other closely. The small difference between them is the condenser approach temperature (TCA).

**TCI and TCO (r = +0.96):** Condenser water inlet and outlet temperatures are strongly correlated because the outlet temperature is directly determined by the inlet temperature plus the heat absorbed from the refrigerant.

**TCA and TRC_sub (r = +0.93):** Condenser approach temperature and subcooling are strongly correlated. Both relate to condenser performance — when the condenser is working well, both values are stable; when fouling occurs, both change together.

**TEO and T_suc (r = +0.92), TRE and T_suc (r = +0.92):** Suction temperature is closely tied to both the evaporator water outlet and the refrigerant evaporating temperature, because the suction line carries refrigerant that has just evaporated in the evaporator.

**kW and Tsh_dis (r = -0.86):** Compressor power and discharge superheat are negatively correlated. When the compressor works harder (higher kW), the discharge superheat tends to decrease because higher mass flow rates reduce superheat.

**Implication for modelling:** Highly correlated features carry redundant information. For the first iteration we keep all 16 features and let the model (via SHAP feature importance in Phase 3) determine which are truly valuable. Tree-based models like XGBoost handle correlated features well without performance degradation.

## 2.5 Good Indicators — Feature Differences by Fault Type

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()
plot_order = ['Normal','CF','FWC','FWE','RL','RO','EO','NC']
for i, col in enumerate(FEATURE_COLS):
    sns.boxplot(data=df, x='fault_short', y=col, ax=axes[i],
               palette='husl', fliersize=1.5, order=plot_order)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_xlabel('')
fig.suptitle('Feature Distributions by Fault Type\n(Features that differ between classes = good indicators)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretation — Good Indicators (Boxplots)

These boxplots are the most important visualization in this EDA because they directly show which features can help the model **distinguish between fault types**. A feature is a good indicator if its distribution visibly shifts for one or more fault types compared to Normal.

**Strong indicators (clear visual separation):**

- **TCA (Condenser Approach Temperature):** Shows a clear upward shift for Condenser Fouling (CF). This is the single most diagnostic feature for CF — when tubes are plugged, heat transfer degrades and the approach temperature increases. This is physically expected and well-documented in HVAC engineering.

- **Tsh_dis (Discharge Superheat):** Has a dramatically wider distribution for Refrigerant Leak (RL). When refrigerant charge is low, the compressor handles more vapour and less liquid, causing discharge superheat to spike. This is one of the primary diagnostic indicators used by HVAC technicians in the field.

- **TRC_sub (Condenser Subcooling):** Drops for Refrigerant Leak (RL) and rises for Refrigerant Overcharge (RO). Subcooling directly reflects how much liquid refrigerant is available in the condenser — less charge means less subcooling, more charge means more. This makes it a strong bidirectional indicator.

- **PO_net (Net Oil Pressure):** Shows a visible increase for Excess Oil (EO), which makes direct physical sense — more oil in the system increases oil pump differential pressure.

**Moderate indicators:**

- **kW (Compressor Power):** Shows subtle shifts for some faults. Reduced water flow faults (FWC, FWE) tend to reduce cooling load, which reduces compressor power.
- **TEO and T_suc:** Shift downward for some faults, particularly reduced evaporator water flow (FWE), because less water flow means the chiller overcools the water that does pass through.

**Weak indicators:**

- **TR_dis and TO_sump:** Show relatively little variation across fault types. Their distributions overlap heavily between Normal and most faults. However, they may still contribute in combination with other features.

**Key takeaway:** No single feature can distinguish all 7 fault types from normal operation. Each fault affects a different subset of features. This confirms that a **multi-feature model** (rather than simple threshold rules) is needed.

In [ ]:
# Quantitative comparison: mean values Normal vs Fault
normal_means = df[df['label'] == 1][FEATURE_COLS].mean()
fault_means = df[df['label'] != 1][FEATURE_COLS].mean()
comparison = pd.DataFrame({
    'Normal (mean)': normal_means.round(2),
    'Fault (mean)': fault_means.round(2),
    'Difference': (fault_means - normal_means).round(2),
    'Diff %': ((fault_means - normal_means) / normal_means * 100).round(1),
}).sort_values('Diff %', key=abs, ascending=False)
print("Feature comparison: Normal vs Fault (sorted by largest difference)")
print("=" * 70)
print(comparison.to_string())

### Interpretation — Quantitative Indicator Ranking

**TCA** leads with a +20.9% increase during faults, driven primarily by the condenser fouling fault. **TRC_sub** follows at +13.5%, sensitive to refrigerant charge faults. **PO_net** at +4.5% reflects oil system changes.

Features at the bottom (TR_dis at 0.0%, TRC at -0.0%) show virtually no difference *on average*. However, they may differ for specific fault types even if the overall average cancels out. The per-fault boxplots above are therefore more informative than this aggregate table.

## 2.6 Outlier Assessment

In [ ]:
outlier_info = []
for col in FEATURE_COLS:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    outlier_info.append({'Feature': col, 'Outliers': n_out, 'Pct': round(n_out/len(df)*100, 2)})
print("Outliers per feature (IQR x 1.5 method):")
print(pd.DataFrame(outlier_info).sort_values('Pct', ascending=False).to_string(index=False))

### Interpretation — Outliers

**TEO** has the highest outlier count at 28.2%. This is not a data quality problem. The IQR method identifies statistical outliers relative to the overall distribution, but in a fault detection dataset, many of these outliers are actually fault signatures. When a chiller fault occurs, TEO (chilled water outlet temperature) deviates from its normal range because the chiller cannot maintain the target temperature.

The same reasoning applies to **T_suc** (12.3%), **PO_net** (9.8%), and **Tsh_suc** (5.6%). These are among the features identified as good fault indicators — their outliers *are* the faults.

**Decision: We do NOT remove outliers.** In a standard data cleaning task, outlier removal makes sense because extreme values are likely errors. In a fault detection context, removing outliers would literally mean removing the fault events we are trying to detect. The only outliers worth removing would be sensor malfunctions (e.g., a sensor reporting exactly 0 or -999), and no such values are present in this laboratory dataset.

## 2.7 EDA Summary

1. **Data quality is high:** Zero missing values, 3 harmless duplicates, all values physically realistic.
2. **Class distribution is moderately imbalanced:** 4,000 normal vs 7,000 fault samples (opposite of real-world).
3. **Strong fault indicators exist:** TCA, TRC_sub, Tsh_dis, and PO_net show clear separation between normal and fault types.
4. **Correlations are physically meaningful:** High correlations (e.g., TCO-TRC at r=0.97) reflect known thermodynamic relationships.
5. **No single feature separates all faults:** Each fault affects a different combination of features, confirming the need for a multi-feature ML model.
6. **Outliers are fault signatures, not errors:** Preserved intentionally.

---
# Part 3: Data Preparation

Steps: binary target creation, duplicate removal, **three-way split (train / validation / test)**, scaling, and saving for Phase 3.

### Why a three-way split and not just train/test?

A simple train/test split is not enough when we also need to tune hyperparameters and compare models. If we use the test set for tuning decisions, the "test" performance is no longer an honest estimate of how the model will do on unseen data — we leak information about the test set into the model through our choices.

The correct setup is **three sets**:
- **Train (~64%):** Fit the model parameters (what the algorithm learns internally)
- **Validation (~16%):** Tune hyperparameters, compare models, make design decisions
- **Test (~20%):** Final honest evaluation, used ONCE at the end

This also applies to scaling: the scaler is fit on training data ONLY, then applied to validation and test without looking at their statistics. If we fit the scaler on the full dataset first, we leak the test distribution into training — a subtle but real form of data leakage.


In [ ]:
# 3.1 Create binary target
df['is_fault'] = (df['label'] != 1).astype(int)
print("Binary target created:")
print(f"  Normal (is_fault=0): {(df['is_fault']==0).sum():,}")
print(f"  Fault  (is_fault=1): {(df['is_fault']==1).sum():,}")
print(f"  Fault ratio: {df['is_fault'].mean():.1%}")

In [ ]:
# 3.2 Remove duplicates
n_before = len(df)
df = df.drop_duplicates(subset=FEATURE_COLS + ['label']).reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicate rows. Dataset: {n_before:,} -> {len(df):,} rows")

In [ ]:
# 3.3 Three-way stratified split: 64% train / 16% val / 20% test
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS].values
y_binary = df['is_fault'].values
y_multi = df['label'].values

# First split: separate out the final test set (20%)
X_trainval, X_test, y_trainval, y_test, y_trainval_multi, y_test_multi = train_test_split(
    X, y_binary, y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_binary,
)

# Second split: from the remaining 80%, carve out validation (20% of 80% = 16% of total)
X_train, X_val, y_train, y_val, y_train_multi, y_val_multi = train_test_split(
    X_trainval, y_trainval, y_trainval_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_trainval,
)

total = len(df)
print("THREE-WAY STRATIFIED SPLIT")
print("=" * 50)
print(f"Train:      {X_train.shape[0]:,} samples ({X_train.shape[0]/total:.1%}) — fault ratio: {y_train.mean():.1%}")
print(f"Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/total:.1%}) — fault ratio: {y_val.mean():.1%}")
print(f"Test:       {X_test.shape[0]:,} samples ({X_test.shape[0]/total:.1%}) — fault ratio: {y_test.mean():.1%}")
print(f"\nStratification ensures identical fault ratios across all three sets.")
print(f"Validation is used for hyperparameter tuning and model selection.")
print(f"Test is held out and used ONLY for final evaluation in Phase 3.")


In [ ]:
# 3.4 Scale features — fit on train ONLY, apply to val and test
# RobustScaler uses median and IQR (robust to the outliers we found in EDA)
# CRITICAL: fit on training data only to prevent data leakage into val/test
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit AND transform on train
X_val_scaled = scaler.transform(X_val)           # Transform only on val
X_test_scaled = scaler.transform(X_test)         # Transform only on test

print("RobustScaler fitted on training data only (no data leakage to val or test).")
print(f"\nExample — feature 'kW' (col 4):")
print(f"  Train — Before: mean={X_train[:, 4].mean():.2f}, std={X_train[:, 4].std():.2f}")
print(f"  Train — After:  mean={X_train_scaled[:, 4].mean():.2f}, std={X_train_scaled[:, 4].std():.2f}")
print(f"  Val   — After:  mean={X_val_scaled[:, 4].mean():.2f}, std={X_val_scaled[:, 4].std():.2f}")
print(f"  Test  — After:  mean={X_test_scaled[:, 4].mean():.2f}, std={X_test_scaled[:, 4].std():.2f}")


In [ ]:
# 3.5 Verify scaling visually
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.DataFrame(X_train, columns=FEATURE_COLS).boxplot(
    ax=axes[0], vert=True, patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Before Scaling', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=90, labelsize=8)
pd.DataFrame(X_train_scaled, columns=FEATURE_COLS).boxplot(
    ax=axes[1], vert=True, patch_artist=True, boxprops=dict(facecolor='lightgreen'))
axes[1].set_title('After RobustScaler', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=90, labelsize=8)
fig.suptitle('Effect of Scaling on Feature Ranges', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Interpretation — Scaling

The left boxplot shows raw feature ranges. Features like TR_dis (79-156 deg F) and TO_sump (109-123 deg F) have much larger absolute values than TEA (0-12 deg F) and TCA (0-11 deg F). Without scaling, a distance-based or gradient-based algorithm would weight the high-magnitude features more heavily simply because their numbers are bigger — not because they are more informative.

After RobustScaler, all features are centered around 0 with comparable spread. The medians are at 0 and the IQR is approximately 1. This ensures the model treats all features equally and learns their importance from the data patterns, not from the unit of measurement.

In [ ]:
# 3.6 Class imbalance for modelling
n_normal = (y_train == 0).sum()
n_fault = (y_train == 1).sum()
ratio = n_normal / n_fault
print("CLASS IMBALANCE (training set)")
print("=" * 40)
print(f"Normal: {n_normal:,}  ({n_normal/(n_normal+n_fault)*100:.1f}%)")
print(f"Fault:  {n_fault:,}  ({n_fault/(n_normal+n_fault)*100:.1f}%)")
print(f"Ratio:  {ratio:.2f}:1")
print(f"\nFor XGBoost: scale_pos_weight = {ratio:.2f}")

In [ ]:
# 3.7 Save prepared data for Phase 3 — all three sets
import pickle

out_dir = Path("data/prepared")
out_dir.mkdir(parents=True, exist_ok=True)

# Scaled features
np.save(out_dir / "X_train_scaled.npy", X_train_scaled)
np.save(out_dir / "X_val_scaled.npy",   X_val_scaled)
np.save(out_dir / "X_test_scaled.npy",  X_test_scaled)

# Raw features (for local interpretations, e.g. SHAP with real sensor values)
np.save(out_dir / "X_train_raw.npy", X_train)
np.save(out_dir / "X_val_raw.npy",   X_val)
np.save(out_dir / "X_test_raw.npy",  X_test)

# Binary targets
np.save(out_dir / "y_train.npy", y_train)
np.save(out_dir / "y_val.npy",   y_val)
np.save(out_dir / "y_test.npy",  y_test)

# Multiclass targets (for future 7-class fault identification work)
np.save(out_dir / "y_train_multi.npy", y_train_multi)
np.save(out_dir / "y_val_multi.npy",   y_val_multi)
np.save(out_dir / "y_test_multi.npy",  y_test_multi)

# Supporting artifacts
pd.Series(FEATURE_COLS).to_csv(out_dir / "feature_names.csv", index=False, header=['feature'])
pd.DataFrame(list(LABEL_MAP.items()), columns=['label','fault_name']).to_csv(
    out_dir / "label_map.csv", index=False)
with open(out_dir / "scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)
df.to_csv(out_dir / "df_clean.csv", index=False)

data_dict = []
for col in FEATURE_COLS:
    data_dict.append({'Feature': col, 'Type': 'float64',
        'Min': round(df[col].min(), 2), 'Max': round(df[col].max(), 2),
        'Mean': round(df[col].mean(), 2), 'Std': round(df[col].std(), 2), 'Missing': 0})
pd.DataFrame(data_dict).to_csv(out_dir / "data_dictionary.csv", index=False)

print("All prepared data saved to data/prepared/:")
for f in sorted(out_dir.glob("*")):
    print(f"   {f.name:30s} ({f.stat().st_size / 1024:.1f} KB)")


---
# Phase 2 Summary

## Data Collection
- Dataset downloaded from Figshare (DOI: 10.6084/m9.figshare.28435232.v1) on 24 March 2026
- Original source: ASHRAE RP-1043 (Comstock & Braun, 1999, Purdue University)
- 11,000 samples, 16 fault-sensitive features, 8 classes (1 normal + 7 fault types), all severity level 1
- License: CC-BY 4.0, no GDPR concerns (equipment sensor data only)
- Stored locally with Git version control; naming convention documented

## Data Understanding
- **Quality:** Zero missing values, 3 duplicates (removed), all values physically realistic
- **Distribution:** 4,000 normal + 7,000 fault samples (0.57:1 — opposite of real-world imbalance)
- **Strong indicators:** TCA (+20.9%), TRC_sub (+13.5%), Tsh_dis (wide range for RL), PO_net (+4.5%)
- **Correlations:** 7 highly correlated pairs, all physically explainable (e.g., TCO-TRC r=0.97)
- **Outliers:** TEO 28%, T_suc 12%, PO_net 10% — fault signatures, preserved intentionally
- **Key finding:** No single feature separates all faults; multi-feature ML model required

## Data Preparation
- Binary target created: Normal (0) vs Fault (1)
- 3 duplicate rows removed
- **Three-way stratified split:** Train (~64%) / Validation (~16%) / Test (~20%)
- RobustScaler fitted on **training data only** (no leakage into validation or test)
- All artifacts saved to `data/prepared/` for Phase 3

## Data Leakage Prevention
Two layers of leakage protection:
1. **Split before scaling:** The scaler sees only training data; it never learns statistics from validation or test.
2. **Held-out test set:** The test set is used once, at the very end of Phase 3, for final honest evaluation. All tuning, model selection, and comparison happen on validation.

## Ready for Phase 3
Next: train baseline and additional models on the training set, tune hyperparameters and compare models on the validation set, then evaluate the final selected model once on the test set. Explain predictions with SHAP.
